<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB19_Case_Study_CLIWOC_Nationality_from_Ship_Routes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB19 · Class 19 — Case Study: CLIWOC Historical Ship Logbooks, Classifying Nationality from Routes**

## Block 4: Proyectos — Case Studies (continued)

`NB18` used real 21st-century weather data. This case study reaches back much further: real 18th-century ship logbooks, digitized for climate research, used here to ask a different kind of question — can a ship's **route and timing alone** reveal which nation it sailed for?

**The real-world problem**: across the mid-to-late 1700s, the British, Dutch, Spanish, and French each ran established maritime trade networks tied to their colonial possessions — Spain's transatlantic and Manila galleon routes, the Dutch VOC's Cape route to the East Indies, British East India Company and Atlantic trade, French Caribbean and Indian Ocean trade. If those networks were real and geographically distinct, a model should be able to recover "which nation" from nothing but *where* and *when* a ship was — `a genuine test of whether real historical trade geography is learnable from data, not an arbitrary classroom label`.

We download the real dataset live from Kaggle, so the exact rows and years available may vary slightly depending on the current release — this notebook is deliberately written to **discover** the real data's structure and time span rather than assume fixed numbers, exactly the habit `NB02`'s "load → inspect" workflow was built around.

Following `NB18`'s corrected structure, this class again has **two parts**: **Part A** (Sections 8–9) validates a modeling approach with a standard random split; **Part B** (Section 10) is the real test — training only on the earlier ~80% of available years and predicting nationality for the **later years the model has never seen**, checking whether these trade-route patterns actually held stable across time, or shifted.

### Learning objectives

By the end of this class, students will be able to:
- Explain why "route → nationality" is a real, historically grounded question, not an arbitrary label.
- Download a real dataset from Kaggle using secure, session-only credentials.
- Work with a real, messy historical dataset — including discovering its actual structure and class balance, rather than assuming them in advance.
- Visualize class-separated geographic data on a real map and use it to sanity-check a modeling premise before training anything.
- Handle class imbalance with a naive baseline and `class_weight="balanced"`.
- Design a temporal holdout that matches the real question being asked (generalizing across years, not being told the "answer" it already saw).

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap, today's roadmap | 5 min | Theory |
| 2 | What is CLIWOC, and why "route → nationality" is a real question | 10 min | Theory |
| 3 | Setting up Kaggle API access | 5 min | Practice |
| 4 | Downloading the real dataset | 5 min | Practice |
| 5 | Exploring the data: columns, nationalities, routes on a real map | 15 min | Practice |
| 6 | Preparing features and confronting real class imbalance | 15 min | Theory + Practice |
| 7 | Applying `NB17`'s decision framework | 5 min | Theory + Practice |
| 8 | Part A: training and comparing models (methodology validation) | 15 min | Practice |
| 9 | Part A: evaluation | 20 min | Practice |
| 10 | Part B: a genuine test — forecasting held-out years | 10 min | Practice |
| 11 | Interpreting Part B, and connecting it to real history | 10 min | Practice |
| 12 | Summary, homework, next class | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.


---

## 1. Recap: where we are

- **`NB18`**: real ECMWF weather data, a genuine held-out-year forecast, and the lesson that a fair generalization test has to hold out the *right* variable (a year, not a season, when the target is seasonal).
- **`NB19`** (today): a different kind of real data — historical, sparse, imbalanced — and the same discipline applied to a temporal holdout that matches *this* problem's real question.

---

## 2. What is CLIWOC, and why "route → nationality" is a real question

**[CLIWOC](https://en.wikipedia.org/wiki/CLIWOC)** (Climatological Database for the World's Oceans) was a real research project that converted historical ships' logbooks — British, Dutch, French, and Spanish, 1750–1850 — into a standardized digital database, originally to reconstruct historical climate and wind patterns from centuries of daily noon observations. That means every row in this dataset is a **real entry a real ship's officer wrote down** at sea, up to 275 years ago.

Today's question uses the same data for a different purpose: each of the four nations ran distinct, real trade networks shaped by their colonial territories and monopolies — Spain's transatlantic and Pacific galleon routes, the Dutch East India Company's Cape-of-Good-Hope route to Indonesia, British Atlantic and Indian Ocean trade, French Caribbean and Indian Ocean trade. If those networks were geographically real and distinct (which real maritime history says they were), a model trained only on **where** and **when** a logbook entry was recorded should be able to recover **which nation** wrote it — `genuine historical geography, learnable from data, not an arbitrary label invented for a homework problem`.

---

## 3. Setting up Kaggle API access

The full CLIWOC database — over 287,000 real logbook entries, 141 columns, spanning 1750–1850 — is published on Kaggle. Downloading it needs a free Kaggle account and API key:

1. Go to [kaggle.com/settings](https://www.kaggle.com/settings), scroll to **API**, and click **Create New Token** — this downloads a `kaggle.json` file to your computer.
2. Run the cell below; it will show a **file picker** — select the `kaggle.json` you just downloaded.

The file's bytes go straight into this Colab runtime's private storage and are never written into this notebook's saved code or output — a real credential should never sit as plain, committed text in a cell, which is exactly how this repository leaked a real Kaggle key earlier in this course's history.

In [ ]:
%pip install -q kaggle

import os
from google.colab import files

print("Select your kaggle.json file:")
uploaded = files.upload()

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kaggle_json_path = os.path.expanduser("~/.kaggle/kaggle.json")
with open(kaggle_json_path, "wb") as f:
    f.write(next(iter(uploaded.values())))
os.chmod(kaggle_json_path, 0o600)

print("Kaggle credentials saved for this session (not stored anywhere in this notebook).")

---

## 4. Downloading the real dataset

In [ ]:
!kaggle datasets download -d cwiloc/climate-data-from-ocean-ships --force --unzip

import os

csv_files = [f for f in os.listdir(".") if f.lower().endswith(".csv")]
print("CSV files in the downloaded archive:", csv_files)

---

## 5. Exploring the data: columns, nationalities, routes on a real map

`NB02`'s starting questions, on real 18th-century data this time. The archive's main file is normally `CLIWOC15.csv` — load it and look at its real columns first, exactly like `NB02`'s "load → inspect" habit, since with 180 real columns there's no point guessing:

In [ ]:
import pandas as pd

raw = pd.read_csv("CLIWOC15.csv", low_memory=False)
print(raw.shape)
raw.columns.tolist()

The columns worth today's four-nation route question: `Lon3`/`Lat3` (a cleaned decimal position, among several position representations in this dataset), `Year`, and `Nationality`. Keep only rows where all four are present, and restrict to the four well-represented navies this class is about — matching on the values case-insensitively, since we haven't confirmed their exact casing in this release yet:

In [ ]:
print(raw["Nationality"].value_counts())

Filter to real, complete rows for our four target nations, and rename the position columns to plain `longitude`/`latitude` so the rest of the notebook doesn't need to know the original column names:

In [ ]:
target_nations = ["BRITISH", "DUTCH", "SPANISH", "FRENCH"]
nat_upper = raw["Nationality"].astype(str).str.upper().str.strip()

cliwoc = raw.loc[nat_upper.isin(target_nations), ["Lon3", "Lat3", "Year", "Nationality"]].copy()
cliwoc["Nationality"] = nat_upper[nat_upper.isin(target_nations)]
cliwoc = cliwoc.rename(columns={"Lon3": "longitude", "Lat3": "latitude"}).dropna()

print(cliwoc.shape)
cliwoc.head()

Now the real, discovered summary — no assumptions, just what this download actually contains:

In [ ]:
print(cliwoc["Nationality"].value_counts())
print()
print("Year range:", cliwoc["Year"].min(), "-", cliwoc["Year"].max())

**Read your own output**: are the four classes close to balanced, or is one nation noticeably rarer than the others? Real archive coverage (how many logbooks from each nation survived and were digitized) `rarely produces a perfectly balanced dataset — a real-world imbalance worth carrying forward`, in the same spirit `NB13` flagged for its own real, self-derived labels.

Not just the overall totals — check whether each nation's logbooks span the *same* years, or whether some nations' records were digitized for a narrower period than others. This isn't idle curiosity: it directly affects Part B's held-out-year design later on.

In [ ]:
print(cliwoc.groupby("Nationality")["Year"].agg(["min", "max", "count"]))

**Read your own output**: if one or more nations' logbooks stop well before the dataset's latest year, a plain "last 20% of all years" cutoff would end up testing almost entirely on whichever nation's records extend furthest — an artifact of uneven digitization coverage, not a genuine test of route stability. Section 10 picks its cutoff accounting for this, restricting candidate years to ones where every nation still has real entries.

Before training anything, test the actual premise visually: does each nation's real logbook data trace a geographically distinct pattern?

In [ ]:
%pip install -q cartopy

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

sample = cliwoc.sample(min(5000, len(cliwoc)), random_state=42)
colors = {"BRITISH": "tab:blue", "DUTCH": "tab:orange", "SPANISH": "tab:green", "FRENCH": "tab:red"}

fig = plt.figure(figsize=(12, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines(resolution="110m")
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.add_feature(cfeature.LAND, facecolor="whitesmoke")

for nation, color in colors.items():
    subset = sample[sample["Nationality"] == nation]
    ax.scatter(subset["longitude"], subset["latitude"], s=4, alpha=0.5,
               label=nation, color=color, transform=ccrs.PlateCarree())

ax.legend(markerscale=3, loc="lower left")
year_lo, year_hi = int(cliwoc["Year"].min()), int(cliwoc["Year"].max())
ax.set_title(f"Real CLIWOC logbook positions by nationality, {year_lo}-{year_hi} ({len(sample):,}-entry sample)")
plt.show()

**Read your own map**: can you see the Spanish transatlantic/Pacific routes, the Dutch Cape-of-Good-Hope corridor toward Indonesia, the British Atlantic and Indian Ocean presence? If these four colors separate into visually distinct regions, `that is real, direct evidence the "route reveals nationality" premise holds` *before* we ever train a model — the map itself is doing genuine exploratory data analysis, `NB02`-style, on a real historical question.

---

## 6. Preparing features and confronting real class imbalance

Features: `latitude`, `longitude` (where) and `Year` (when). Target: `Nationality`. (`CLIWOC15.csv` doesn't reliably carry a clean `Month` column across all entries the way `Year` does, so we keep the feature set to what we've confirmed is complete.)

In [ ]:
feature_cols = ["latitude", "longitude", "Year"]
X = cliwoc[feature_cols]
y = cliwoc["Nationality"]

print(y.value_counts(normalize=True).round(3))

Seeing the imbalance as a bar chart makes it easier to judge at a glance:

In [ ]:
counts = y.value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.index, counts.values, color="steelblue")
ax.set_ylabel("Number of entries")
ax.set_title("Class balance across the four nations")
plt.show()


**Read the cell above's output**: whichever nation came out rarest, a model that ignores it entirely could still score high overall accuracy — `NB07`'s original imbalance warning, now with real numbers behind it. We'll use two concrete tools against this: a **naive baseline** to know what "cheating by ignoring the minority class" would actually score, and scikit-learn's `class_weight="balanced"` option, which up-weights minority-class errors during training instead of treating every mistake equally.

---

## 7. Applying `NB17`'s decision framework

1. **Labels?** Yes — `Nationality` is real and known for every entry.
2. **Data shape?** Tabular — three simple numeric features per row.
3. **Data volume?** Look at `X.shape[0]` above — likely tens of thousands of rows or more, comfortably in the range where either classical ML or a neural network could work; `NB17`'s framework doesn't strongly favor one here the way it did for `NB10`'s 308-row yacht data.

Given the framework doesn't push hard in either direction this time, we'll use a classical ensemble (fast, interpretable, easy to weight for imbalance) — but note for yourself that this is a case where trying a small MLP (`NB11` style) as homework is a genuinely open question, not a foregone conclusion.

---

## 8. Part A: training and comparing models (methodology validation)

A random, stratified split first — the same kind of "validate the approach" step as `NB18`'s Part A:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)

Compare the naive baseline against a class-weighted Random Forest:

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_scaled, y_train)
print("Naive baseline (always predict the majority class) accuracy:", round(baseline.score(X_test_scaled, y_test), 3))

rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
rf.fit(X_train_scaled, y_train)
print("Random Forest accuracy:", round(rf.score(X_test_scaled, y_test), 3))

---

## 9. Part A: evaluation

Overall accuracy hides how each individual nation is doing — a full report matters more here than usual, given the imbalance:

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = rf.predict(X_test_scaled)
print(confusion_matrix(y_test, y_pred, labels=rf.classes_))
print()
print(classification_report(y_test, y_pred, labels=rf.classes_))

A visual confusion matrix, the same way `NB07`/`NB11` displayed it:

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, labels=rf.classes_), display_labels=rf.classes_).plot(cmap="Blues", ax=ax, xticks_rotation=45)
plt.show()


**Try it yourself**: pull each nation's individual recall out of the classification report and plot it — is the imbalance visible directly in per-class recall, not just in the raw counts from Section 6?

In [ ]:
report_dict = classification_report(y_test, y_pred, labels=rf.classes_, output_dict=True)
recalls = {nation: report_dict[nation]["recall"] for nation in rf.classes_}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(recalls.keys(), recalls.values(), color="darkorange")
ax.set_ylabel("Recall")
ax.set_title("Part A: recall per nation")
ax.set_ylim(0, 1)
plt.show()


**Read your own report**: is the rarest nation's recall noticeably lower than the other three, even with `class_weight="balanced"`? That would be an honest, expected finding given how little data exists for it — a real limitation of the underlying archive, not a modeling mistake to fix away.

---

## 10. Part B: a genuine test — forecasting held-out years

Part A's random split mixes entries from every era into both train and test — a fair methodology check, but not a real test of whether these route patterns *held up over time*. The genuine question: train only on earlier years, then predict nationality for **years the model has never seen**, exactly the same discipline `NB18` used for its held-out year, applied to the variable that actually matters for *this* question (time period, not season).

Section 5's coverage check may have already shown some nations' logbooks stop years before others'. To keep this a fair test rather than one that quietly collapses into "predict whichever nation's archive runs longest," the cutoff below is chosen only from **years where every nation still has real entries** — the intersection of each nation's covered years, not the full dataset's year range.

One more adjustment, for the same reason `NB18` dropped its own `year` feature before pooling multiple training years: `Year` was a legitimate input for Part A (train and test both drawn from the same range), but Part B's whole point is testing on years the model has never seen — feeding `Year` as a raw number into a tree-based model that then has to score rows with `Year` values *past* every threshold it ever split on is a real limitation of how tree ensembles extrapolate, not a fair test of whether **routes** generalize. Part B trains on `latitude`/`longitude` only, so any accuracy drop reflects the geography itself, not the model failing to extrapolate a number it was never built to extrapolate.

CLIWOC's real coverage here turns out to span **centuries**, not decades (this is genuinely why the notebook discovers its own year range instead of assuming one). Testing on "every year from the cutoff to the end of the archive" `would silently compare early-colonial trade geography against everything up to the mid-19th century` — including the Napoleonic Wars, the 1799 dissolution of the Dutch VOC, and the Spanish American wars of independence, a full reshuffling of colonial trade that has nothing to do with whether routes are stable in the *near* term. So Part B holds out a **bounded window right after the cutoff** — comparable in spirit to `NB18`'s single held-out year — not an open-ended "rest of history".

Last check: some nations' real logbook presence is concentrated in specific historical bursts (a particular expedition, a particular decade of active trade) rather than spread evenly across two centuries. A nation with only a handful of real entries in this specific window can't support a fair classification test no matter how the model is tuned — that's a genuine data-sparsity limit, not something to fix with a different split. So Part B automatically **drops any nation below a minimum real share of the test window**, and says so explicitly, rather than forcing a comparison that a handful of rows can't honestly support:

In [ ]:
feature_cols_era = ["latitude", "longitude"]
TEST_WINDOW_YEARS = 10
MIN_TEST_SHARE = 0.05  # a nation needs at least 5% of the test window's real rows

years_per_nation = [set(cliwoc.loc[cliwoc["Nationality"] == nat, "Year"]) for nat in cliwoc["Nationality"].unique()]
common_years = sorted(set.intersection(*years_per_nation))

if len(common_years) < 5:
    raise ValueError(
        "Fewer than 5 years have entries from every nation -- the four nations' "
        "logbook coverage barely overlaps in time, so a fair year-based holdout "
        "isn't possible here. Inspect Section 5's per-nation year ranges above "
        "to see which nation's coverage is the bottleneck."
    )

cutoff_year = common_years[int(len(common_years) * 0.8)]
window_end_year = cutoff_year + TEST_WINDOW_YEARS

train_era = cliwoc[cliwoc["Year"] < cutoff_year]
test_era = cliwoc[(cliwoc["Year"] >= cutoff_year) & (cliwoc["Year"] < window_end_year)]

if len(test_era) == 0:
    raise ValueError(
        f"No entries fall in the {cutoff_year}-{window_end_year - 1} test window -- "
        "try a larger TEST_WINDOW_YEARS, or inspect Section 5's per-nation year "
        "ranges to see where coverage actually thins out."
    )

nation_share = test_era["Nationality"].value_counts(normalize=True)
stable_nations = sorted(nation_share[nation_share >= MIN_TEST_SHARE].index)
dropped_nations = sorted(set(cliwoc["Nationality"].unique()) - set(stable_nations))

if dropped_nations:
    print(f"Dropping {dropped_nations} from Part B: under {MIN_TEST_SHARE:.0%} of the "
          f"{cutoff_year}-{window_end_year - 1} window's real entries -- too sparse in "
          "this specific era for a fair test, even though Part A's full random split "
          "above had enough data for them.")

train_era = train_era[train_era["Nationality"].isin(stable_nations)]
test_era = test_era[test_era["Nationality"].isin(stable_nations)]

X_train_era, y_train_era = train_era[feature_cols_era], train_era["Nationality"]
X_test_era, y_test_era = test_era[feature_cols_era], test_era["Nationality"]

print(f"Held-out cutoff year: {cutoff_year} (chosen from {len(common_years)} years common to all four nations)")
print(f"Part B nations: {stable_nations}")
print(f"Train (years before {cutoff_year}):", X_train_era.shape)
print(f"Test  (years {cutoff_year}-{window_end_year - 1}):", X_test_era.shape)
print(y_test_era.value_counts(normalize=True).round(3))

Train a fresh model — this must **not** reuse the Part A model, which already saw some held-out-era rows in its own training split:

In [ ]:
scaler_era = StandardScaler()
X_train_era_scaled = scaler_era.fit_transform(X_train_era)
X_test_era_scaled = scaler_era.transform(X_test_era)

rf_era = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
rf_era.fit(X_train_era_scaled, y_train_era)

y_pred_era = rf_era.predict(X_test_era_scaled)
print(classification_report(y_test_era, y_pred_era, labels=rf_era.classes_))

**Try it yourself**: don't just eyeball two printed reports — pull each nation's recall from both Part A and Part B into one table, side by side.

In [ ]:
report_dict_a = classification_report(y_test, y_pred, labels=rf.classes_, output_dict=True)
report_dict_b = classification_report(y_test_era, y_pred_era, labels=rf_era.classes_, output_dict=True, zero_division=0)

recall_comparison = pd.DataFrame({
    "Part A recall": {nat: report_dict_a[nat]["recall"] for nat in rf.classes_},
    "Part B recall": {nat: report_dict_b[nat]["recall"] for nat in rf_era.classes_},
})
recall_comparison.round(3)


---

## 11. Interpreting Part B, and connecting it to real history

**If Section 10 printed a shorter `Part B nations` list than Part A's four**, that in itself is a real finding worth stating plainly: whichever nation was dropped simply doesn't have enough real logbook entries in this specific window to support a fair test, even though it had plenty of data overall (Part A trained on it fine). Real archives are not evenly distributed across time — some nations' documented presence is concentrated in specific decades, not spread smoothly across centuries.

**Compare this report to Part A's, for the nations that remain.** A meaningful drop would be a genuinely interesting historical finding, not a failure: `it would suggest trade routes shifted right around the cutoff, even within Section 10's short, bounded window`. A drop is expected here — Part A's random split lets the model see every era at once, while Part B genuinely doesn't; a *moderate* drop (not a collapse to near-zero for every class) is a legitimate result, not something to keep tuning away.

Look up the printed `cutoff_year` from the cell above, and check what was happening historically right around it for the nations in this dataset. If your recall/precision numbers differ noticeably **across nations** (not just overall), that's worth investigating per nation, not just as one aggregate score — different nations can be disrupted by different real events in the same years. For example, a cutoff landing around **1789** sits at the start of a genuinely turbulent decade specifically for the **Dutch**: the Patriot Revolution (1780s) and Prussian invasion (1787) destabilized the Dutch Republic's politics, and the French Revolutionary Army's 1795 invasion (creating the Batavian Republic) triggered the terminal financial crisis of the VOC's trading monopoly, which formally dissolved in 1799 — a disruption to Dutch colonial trade specifically, distinct from what British or Spanish shipping faced in the same years. If Dutch predictions come out noticeably worse than the other nations' in your own run, that real history is a plausible explanation; if a different nation is weakest, or your cutoff lands somewhere else entirely, look up what was happening for *that* nation around *that* year instead of reusing this example.

This is exactly the honest use of a genuine holdout: it doesn't just score a model, it can **surface a real historical question worth investigating further** (was there a real shift, and if so, in which nation's routes specifically?) — a hypothesis this notebook raises but does not claim to prove; you would need real historical analysis, not just one accuracy number, to confirm it.

---

## Class summary

- CLIWOC turns 18th-century ship logbooks into real, structured data — real observations, real class imbalance, real historical stakes behind the numbers, downloaded live from Kaggle with secure, session-only credentials.
- "Route reveals nationality" is a genuine historical claim, checked visually on a real map before any model touched the data.
- A naive baseline and `class_weight="balanced"` are two concrete, complementary tools against real class imbalance — used together, not as a substitute for reading the full per-class report.
- Part A (random split) validates a modeling approach; Part B (train on earlier years, test on a short, bounded held-out window right after the cutoff) is the real test — matching both the holdout variable and its time span to the actual question, exactly as `NB18` established.
- A real accuracy drop across a genuine holdout isn't a failure to explain away — it's a legitimate signal worth connecting to real history.

## For the next class

Another Block 4 case study, using real terrain/elevation data — continuing the same pattern: a real dataset, a real question, and a genuine (not just methodological) holdout wherever the question calls for one.

## Homework / Practice Ideas

1. Add `ShipType` (from the raw dataset, encoded) as a fourth feature — does it improve Part A's per-class recall, especially for the rarest nation?
2. Try a small MLP (`NB11` style) on this task, per Section 7's open question — does it beat the Random Forest here, given how much more data this dataset has than most of this course's other classification tasks?
3. Change `TEST_WINDOW_YEARS` in Section 10 (try 3, then 20) — does a shorter window recover more of Part A's accuracy, and does a much longer one collapse further, the way the original open-ended "rest of history" version did?
4. Recompute Part A using `class_weight=None` (the default, unweighted) instead of `"balanced"` — how much does the rarest nation's recall change, and does overall accuracy go up or down?
5. Using the map from Section 5, pick one nation and describe (in a markdown cell) what its real historical trade routes should look like — does the plotted data match your own historical expectation?

> ***As always: a real historical dataset can teach you as much about history as about machine learning — Section 11's interpretation is not optional decoration, it's the actual point of using data this old.***